In [ ]:
import sys
import numpy as np
import yaml

sys.path.append("..")
from src.model.SSVI import fit_ssvi, predict_ssvi
from src.data_generation.data_preperation import (
    grid_from_cfg, generate_surfaces, sample_sparse_points, _split_context_query,
)
from src.evaluation.surface_eval import check_arbitrage_flat


cfg = yaml.safe_load(open('../config.yaml'))
ttms, ks = grid_from_cfg(cfg)
n_full = len(ttms) * len(ks)

In [ ]:
np.random.seed(0)
N_SURFACES = 50
N_CONTEXTS = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 20, 50]

_, _, surfaces = generate_surfaces(cfg, N_SURFACES)

def contexts_for(n_context):
    k_idx, t_idx = sample_sparse_points(ks, ttms, np.full(N_SURFACES, n_context), N_SURFACES)
    return _split_context_query(ks, ttms, surfaces, k_idx, t_idx)

In [ ]:
results = {}
for n_context in N_CONTEXTS:
    train, test = contexts_for(n_context)
    rmses, cal_v, bfly_v, preds = [], [], [], []
    for (X_tr, y_tr), (X_te, y_te) in zip(train, test):
        params, _ = fit_ssvi(X_tr, y_tr, cfg)
        pred = predict_ssvi(params, ttms, ks).ravel()
        rmses.append(np.sqrt(np.mean((pred - y_te) ** 2)))
        c, b = check_arbitrage_flat(cfg, pred)
        cal_v.append(bool(c)); bfly_v.append(bool(b)); preds.append(pred)
    results[n_context] = dict(rmse=np.array(rmses), cal=np.array(cal_v),
                              bfly=np.array(bfly_v), preds=np.array(preds))
    print(f'N_CONTEXT={n_context:>4}  RMSE={np.mean(rmses):.4f}  '
          f'cal_viol={np.mean(cal_v):.2f}  bfly_viol={np.mean(bfly_v):.2f}')